# Lab 02 Extra - Banco de Dados Northwind
**Disciplina:** Extração e Preparação de Dados | **Professor:** Luis Aramis

Este é um notebook extra para praticar SQL com um banco de dados clássico: o **Northwind**.
Ele simula uma importadora/exportadora de alimentos gourmet.

## 1. Setup e Download
Vamos baixar o `northwind.db` e conectar o SQLAlchemy.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
import urllib.request

# Download do northwind.db
if not os.path.exists('northwind.db'):
    # URL do repositório jpwhite3/northwind-SQLite3
    url = 'https://github.com/jpwhite3/northwind-SQLite3/raw/main/dist/northwind.db'
    urllib.request.urlretrieve(url, 'northwind.db')
    print('Banco Northwind baixado com sucesso!')

engine = create_engine('sqlite:///northwind.db')
print('Conexão estabelecida!')

Conexão estabelecida!


## 2. Mapa do Banco
Quais tabelas temos aqui?

In [2]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", engine)

,name
0,Categories
1,sqlite_sequence
2,CustomerCustomerDemo
3,CustomerDemographics
4,Customers
5,Employees
6,EmployeeTerritories
7,Order Details
8,Orders
9,Products


## 3. Consultas Básicas
1. Liste os 5 produtos mais caros (`Products`).
2. Liste todos os clientes (`Customers`) que moram no 'Brazil'.

In [3]:
# Produtos mais caros
df_expensive = pd.read_sql('SELECT * FROM Products ORDER BY UnitPrice DESC LIMIT 5', engine)
df_expensive

,ProductID,ProductName,SupplierID,CategoryID,QuantityPerUnit,UnitPrice,UnitsInStock,UnitsOnOrder,ReorderLevel,Discontinued
0,38,Côte de Blaye,18,1,12 - 75 cl bottles,263.50,17,0,15,0
1,29,Thüringer Rostbratwurst,12,6,50 bags x 30 sausgs.,123.79,0,0,0,1
2,9,Mishi Kobe Niku,4,6,18 - 500 g pkgs.,97.00,29,0,0,1
3,20,Sir Rodney's Marmalade,8,3,30 gift boxes,81.00,40,0,0,0
4,18,Carnarvon Tigers,7,8,16 kg pkg.,62.50,42,0,0,0


In [4]:
# Clientes do Brasil
df_brazil = pd.read_sql("SELECT * FROM Customers WHERE Country = 'Brazil'", engine)
df_brazil

,CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
0,COMMI,Comércio Mineiro,Pedro Afonso,Sales Associate,"Av. dos Lusíadas, 23",Sao Paulo,South America,05432-043,Brazil,(11) 555-7647,None
1,FAMIA,Familia Arquibaldo,Aria Cruz,Marketing Assistant,"Rua Orós, 92",Sao Paulo,South America,05442-030,Brazil,(11) 555-9857,None
2,GOURL,Gourmet Lanchonetes,André Fonseca,Sales Associate,"Av. Brasil, 442",Campinas,South America,04876-786,Brazil,(11) 555-9482,None
3,HANAR,Hanari Carnes,Mario Pontes,Accounting Manager,"Rua do Paço, 67",Rio de Janeiro,South America,05454-876,Brazil,(21) 555-0091,(21) 555-8765
4,QUEDE,Que Delícia,Bernardo Batista,Accounting Manager,"Rua da Panificadora, 12",Rio de Janeiro,South America,02389-673,Brazil,(21) 555-4252,(21) 555-4545
5,QUEEN,Queen Cozinha,Lúcia Carvalho,Marketing Assistant,"Alameda dos Canàrios, 891",Sao Paulo,South America,05487-020,Brazil,(11) 555-1189,None
6,RICAR,Ricardo Adocicados,Janete Limeira,Assistant Sales Agent,"Av. Copacabana, 267",Rio de Janeiro,South America,02389-890,Brazil,(21) 555-3412,None
7,TRADH,Tradição Hipermercados,Anabela Domingues,Sales Representative,"Av. Inês de Castro, 414",Sao Paulo,South America,05634-030,Brazil,(11) 555-2167,(11) 555-2168
8,WELLI,Wellington Importadora,Paula Parente,Sales Manager,"Rua do Mercado, 12",Resende,South America,08737-363,Brazil,(14) 555-8122,None


## 4. JOIN: Pedidos e Clientes
Vamos ver quem fez quais pedidos.
Tabelas: `Orders` e `Customers`.
Chave de ligação: `CustomerID`.

In [5]:
query_orders = 'SELECT o.OrderID, c.CompanyName FROM Orders o JOIN Customers c ON o.CustomerID = c.CustomerID'
pd.read_sql(query_orders, engine).head()

,OrderID,CompanyName
0,10248,Vins et alcools Chevalier
1,10249,Toms Spezialitäten
2,10250,Hanari Carnes
3,10251,Victuailles en stock
4,10252,Suprêmes délices


## 5. JOIN Triplo: Detalhes do Pedido
O que tem dentro do pedido 10248?
Caminho: `OrderDetails` -> `Products`.

In [6]:
query_details = """
SELECT od.OrderID, p.ProductName, od.UnitPrice
FROM "Order Details" od
JOIN Products p ON od.ProductID = p.ProductID
WHERE od.OrderID = 10248
"""
pd.read_sql(query_details, engine)

,OrderID,ProductName,UnitPrice
0,10248,Queso Cabrales,14.0
1,10248,Singaporean Hokkien Fried Mee,9.8
2,10248,Mozzarella di Giovanni,34.8


## 6. Desafio: Total de Vendas por Categoria
Descubra qual Categoria de produtos (`Categories`) gerou mais receita.
Dica: Você vai precisar ligar `Categories` -> `Products` -> `Order Details`.

In [7]:
query_cat = """
SELECT c.CategoryName, SUM(od.UnitPrice * od.Quantity) as TotalReceita
FROM Categories c
JOIN Products p ON c.CategoryID = p.CategoryID
JOIN "Order Details" od ON p.ProductID = od.ProductID
GROUP BY c.CategoryName
ORDER BY TotalReceita DESC
"""
df_cat = pd.read_sql(query_cat, engine)
df_cat

,CategoryName,TotalReceita
0,Beverages,92181842.95
1,Confections,66347544.94
2,Meat/Poultry,64896314.41
3,Dairy Products,58034940.00
4,Condiments,55802774.45
5,Seafood,49931965.52
6,Produce,32706403.90
7,Grains/Cereals,28573512.55
